In [138]:
import pandas as pd
import numpy as np

In [139]:
traffic_data=pd.read_csv('./train.csv')

In [140]:

traffic_data["day"].value_counts()

day
48    69427
49     7872
Name: count, dtype: int64

In [141]:
df=traffic_data.copy()

In [142]:
df["hour"] = df["timestamp"].str.split(":").str[0].astype(int)
df["minute"] = df["timestamp"].str.split(":").str[1].astype(int)

df["time_slot"] = df["hour"] * 4 + df["minute"] // 15

In [143]:
df.tail(4)

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute,time_slot
77295,77295,qp0d4q,49,2:0,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy,2,0,8
77296,77296,qp0d4w,49,2:0,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny,2,0,8
77297,77297,qp0dhw,49,2:0,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny,2,0,8
77298,77298,qp0djq,49,2:0,0.002944,Residential,3,Allowed,Yes,1.322034,Snowy,2,0,8


In [144]:
df=df.drop(columns=['Index','timestamp'],axis=1)

In [145]:
df["Temperature"] = df["Temperature"].fillna(
    df["Temperature"].median()
)

df["RoadType"] = df["RoadType"].fillna("Unknown")

df["Weather"] = df["Weather"].fillna("Unknown")

In [146]:
df["LargeVehicles"] = df["LargeVehicles"].map({
    "Allowed":1,
    "Not Allowed":0
})

df["Landmarks"] = df["Landmarks"].map({
    "Yes":1,
    "No":0
})

In [147]:
df.tail(5)

,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute,time_slot
77294,qp0d4n,49,0.067203,Residential,1,0,0,11.501664,Rainy,2,0,8
77295,qp0d4q,49,0.022859,Residential,3,1,1,14.715254,Foggy,2,0,8
77296,qp0d4w,49,0.141342,Residential,3,1,1,19.678860,Sunny,2,0,8
77297,qp0dhw,49,0.087574,Residential,1,0,0,22.573958,Sunny,2,0,8
77298,qp0djq,49,0.002944,Residential,3,1,1,1.322034,Snowy,2,0,8


In [148]:
df["peak_hour"] = (
    ((df["time_slot"] >= 28) & (df["time_slot"] <= 40))
    |
    ((df["time_slot"] >= 64) & (df["time_slot"] <= 80))
).astype(int)

In [149]:
df=df.drop('minute',axis=1)

In [150]:
df=df.drop(columns=['Weather','Temperature','Landmarks'],axis=1)

In [151]:
df=df.drop('hour',axis=1)

In [152]:
df["geo_road"] = (
    df["geohash"].astype(str)
    + "_"
    + df["RoadType"].astype(str)
)

In [153]:
df["road_time"] = (
    df["RoadType"].astype(str)
    + "_"
    + df["time_slot"].astype(str)
)

In [154]:
geo_freq = traffic_data["geohash"].value_counts()

df["geo_freq"] = df["geohash"].map(geo_freq)

In [155]:
import pygeohash as pgh

In [156]:
def decode_geohash(gh):
    try:
        lat, lon = pgh.decode(gh)
        return pd.Series([lat, lon])
    except:
        return pd.Series([np.nan, np.nan])

df[["latitude", "longitude"]] = df["geohash"].apply(decode_geohash)

In [157]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df.drop("demand", axis=1)
y = df["demand"]

cat_features = [
    "geohash",
    "RoadType",
    "geo_road",
    "road_time",
  
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = CatBoostRegressor(
    iterations=3000,
    depth=10,
    learning_rate=0.03,
    loss_function="RMSE",
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

preds = model.predict(X_test)

print("R2 Score:", r2_score(y_test, preds))

0:	learn: 0.1387999	total: 17.9ms	remaining: 53.8s
200:	learn: 0.0372474	total: 2.7s	remaining: 37.6s
400:	learn: 0.0325475	total: 5.89s	remaining: 38.2s
600:	learn: 0.0298479	total: 9.07s	remaining: 36.2s
800:	learn: 0.0280532	total: 12.3s	remaining: 33.9s
1000:	learn: 0.0267781	total: 15.8s	remaining: 31.6s
1200:	learn: 0.0257399	total: 19s	remaining: 28.4s
1400:	learn: 0.0248803	total: 22.2s	remaining: 25.4s
1600:	learn: 0.0241441	total: 25.6s	remaining: 22.4s
1800:	learn: 0.0234884	total: 29s	remaining: 19.3s
2000:	learn: 0.0229055	total: 32.3s	remaining: 16.1s
2200:	learn: 0.0223682	total: 35.7s	remaining: 13s
2400:	learn: 0.0218745	total: 39.3s	remaining: 9.8s
2600:	learn: 0.0214425	total: 42.9s	remaining: 6.59s
2800:	learn: 0.0210285	total: 46.6s	remaining: 3.31s
2999:	learn: 0.0206343	total: 50.2s	remaining: 0us
R2 Score: 0.9583433068316651


In [162]:
importance = model.get_feature_importance()

feature_imp = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
})

feature_imp.sort_values(
    by="Importance",
    ascending=False
).head(20)

,Feature,Importance
2,RoadType,35.389842
5,time_slot,10.412917
4,LargeVehicles,9.717790
3,NumberofLanes,7.836630
0,geohash,7.687335
9,geo_freq,7.530010
11,longitude,7.297582
10,latitude,5.573087
7,geo_road,4.076574
8,road_time,2.499111


In [163]:
test_data=pd.read_csv('./test.csv')

In [164]:
import pandas as pd
import pygeohash as pgh

test_df = test_data.copy()

# Save Index for submission
test_index = test_df["Index"]

# --------------------------------------------------
# Time Features
# --------------------------------------------------

test_df["hour"] = test_df["timestamp"].str.split(":").str[0].astype(int)
test_df["minute"] = test_df["timestamp"].str.split(":").str[1].astype(int)

test_df["time_slot"] = (
    test_df["hour"] * 4 +
    test_df["minute"] // 15
)

# Peak Hour
test_df["peak_hour"] = (
    ((test_df["time_slot"] >= 28) & (test_df["time_slot"] <= 40))
    |
    ((test_df["time_slot"] >= 64) & (test_df["time_slot"] <= 80))
).astype(int)

# --------------------------------------------------
# Binary Encoding
# --------------------------------------------------

test_df["LargeVehicles"] = test_df["LargeVehicles"].map({
    "Allowed": 1,
    "Not Allowed": 0
})

# --------------------------------------------------
# Missing Values
# --------------------------------------------------

test_df["RoadType"] = test_df["RoadType"].fillna("Unknown")

# --------------------------------------------------
# geo_road
# --------------------------------------------------

test_df["geo_road"] = (
    test_df["geohash"].astype(str)
    + "_"
    + test_df["RoadType"].astype(str)
)

# --------------------------------------------------
# road_time
# --------------------------------------------------

test_df["road_time"] = (
    test_df["RoadType"].astype(str)
    + "_"
    + test_df["time_slot"].astype(str)
)

# --------------------------------------------------
# geo_freq
# Use mapping learned from TRAIN
# --------------------------------------------------

test_df["geo_freq"] = (
    test_df["geohash"]
    .map(geo_freq)
    .fillna(0)
)

# --------------------------------------------------
# Latitude / Longitude
# --------------------------------------------------

def decode_geo(g):
    try:
        lat, lon = pgh.decode(g)
        return pd.Series([lat, lon])
    except:
        return pd.Series([None, None])

test_df[["latitude", "longitude"]] = (
    test_df["geohash"]
    .apply(decode_geo)
)

# --------------------------------------------------
# Drop columns NOT used by model
# --------------------------------------------------

test_df.drop(
    columns=[
        "Index",
        "timestamp",
        "hour",
        "minute",
        "Landmarks",
        "Temperature",
        "Weather"
    ],
    inplace=True,
    errors="ignore"
)

# --------------------------------------------------
# Match training column order
# --------------------------------------------------

test_df = test_df[X.columns]



In [165]:
print(X.columns.tolist())
print(test_df.columns.tolist())

['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'time_slot', 'peak_hour', 'geo_road', 'road_time', 'geo_freq', 'latitude', 'longitude']
['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'time_slot', 'peak_hour', 'geo_road', 'road_time', 'geo_freq', 'latitude', 'longitude']


In [166]:
test_preds = model.predict(test_df)

In [103]:
submission = pd.DataFrame({
    "Index": test_index,
    "demand": test_preds
})

submission.to_csv("submission2.csv", index=False)

In [104]:
data=traffic_data.copy()

In [105]:
data.head(4)

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy


In [106]:
data=data.drop('Index',axis=1)

In [107]:
import pygeohash as pgh

In [110]:
def decode_geohash(gh):
    try:
        lat, lon = pgh.decode(gh)
        return pd.Series([lat, lon])
    except:
        return pd.Series([np.nan, np.nan])

data[["latitude", "longitude"]] = data["geohash"].apply(decode_geohash)

In [111]:
data['hours']=data['timestamp'].str.split(':').str[0].astype(int)

In [112]:
data['min']=data['timestamp'].str.split(':').str[1].astype(int)

In [113]:
data["time_slot"] = data["hours"] * 4 + data["min"] // 15

In [114]:
data["peak_hour"] = (
    ((data["time_slot"] >= 28) & (data["time_slot"] <= 40))
    |
    ((data["time_slot"] >= 64) & (data["time_slot"] <= 80))
).astype(int)

In [115]:
data=data.drop('geohash',axis=1)

In [116]:
data=data.drop('timestamp',axis=1)

In [167]:
data.tail(4)

,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,latitude,longitude,hours,time_slot,peak_hour,RoadType_Residential,RoadType_Street,RoadType_Unknown,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_Unknown,hour_sin,hour_cos
77295,49,0.022859,3,1,1,14.715254,-5.237732,90.807495,2,8,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
77296,49,0.141342,3,1,1,19.678860,-5.237732,90.818481,2,8,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.5,0.866025
77297,49,0.087574,1,0,0,22.573958,-5.237732,90.906372,2,8,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.5,0.866025
77298,49,0.002944,3,1,1,1.322034,-5.237732,90.939331,2,8,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5,0.866025


In [118]:
data['RoadType'].unique()

array([nan, 'Residential', 'Street', 'Highway'], dtype=object)

In [119]:
data["Temperature"] = data["Temperature"].fillna(
    data["Temperature"].median()
)

data["RoadType"] = data["RoadType"].fillna("Unknown")

data["Weather"] = data["Weather"].fillna("Unknown")

In [120]:
data["LargeVehicles"] = data["LargeVehicles"].map({
    "Allowed":1,
    "Not Allowed":0
})

data["Landmarks"] = data["Landmarks"].map({
    "Yes":1,
    "No":0
})

In [121]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop='first',          # avoid dummy variable trap
    handle_unknown='ignore',
    sparse_output=False
)

encoded = ohe.fit_transform(
   data[['RoadType', 'Weather']]
)


In [122]:
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(['RoadType', 'Weather']),
    index=data.index
)

data = pd.concat(
    [
        data.drop(['RoadType', 'Weather'], axis=1),
        encoded_df
    ],
    axis=1
)

In [123]:
data["hour_sin"] = np.sin(2*np.pi*data["hours"]/24)
data["hour_cos"] = np.cos(2*np.pi*data["hours"]/24)

In [124]:
data=data.drop('min',axis=1)

In [49]:
data=data.drop('hours',axis=1)

In [125]:
data.tail(3)

,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,latitude,longitude,hours,time_slot,peak_hour,RoadType_Residential,RoadType_Street,RoadType_Unknown,Weather_Rainy,Weather_Snowy,Weather_Sunny,Weather_Unknown,hour_sin,hour_cos
77296,49,0.141342,3,1,1,19.678860,-5.237732,90.818481,2,8,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.5,0.866025
77297,49,0.087574,1,0,0,22.573958,-5.237732,90.906372,2,8,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.5,0.866025
77298,49,0.002944,3,1,1,1.322034,-5.237732,90.939331,2,8,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.5,0.866025


In [159]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

# Features and Target
x = data.drop("demand", axis=1)
Y = data["demand"]

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    x,
    Y,
    test_size=0.2,
    random_state=42
)

# XGBoost Model
model2 = XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Train
model2.fit(X_train, y_train)

# Predict
preds2 = model2.predict(X_test)

# Evaluate
r2 = r2_score(y_test, preds)

print("R² Score:", r2)

R² Score: 0.9583433068316651


In [161]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": x.columns,
    "Importance": model2.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
).head(20)

,Feature,Importance
10,RoadType_Residential,0.537132
12,RoadType_Unknown,0.133043
2,LargeVehicles,0.131487
11,RoadType_Street,0.108819
1,NumberofLanes,0.028011
3,Landmarks,0.022876
6,longitude,0.007783
5,latitude,0.006281
18,hour_cos,0.005207
17,hour_sin,0.004477


In [168]:
import pandas as pd
import numpy as np
import pygeohash as pgh

test_xgb = test_data.copy()

# Save Index
test_index = test_xgb["Index"]

# -------------------------
# Time Features
# -------------------------

test_xgb["hours"] = (
    test_xgb["timestamp"]
    .str.split(":")
    .str[0]
    .astype(int)
)

test_xgb["minute"] = (
    test_xgb["timestamp"]
    .str.split(":")
    .str[1]
    .astype(int)
)

test_xgb["time_slot"] = (
    test_xgb["hours"] * 4
    + test_xgb["minute"] // 15
)

test_xgb["peak_hour"] = (
    ((test_xgb["time_slot"] >= 28) &
     (test_xgb["time_slot"] <= 40))
    |
    ((test_xgb["time_slot"] >= 64) &
     (test_xgb["time_slot"] <= 80))
).astype(int)

# -------------------------
# Binary Encoding
# -------------------------

test_xgb["LargeVehicles"] = test_xgb["LargeVehicles"].map({
    "Allowed": 1,
    "Not Allowed": 0
})

test_xgb["Landmarks"] = test_xgb["Landmarks"].map({
    "Yes": 1,
    "No": 0
})

# -------------------------
# Missing Values
# -------------------------

test_xgb["RoadType"] = test_xgb["RoadType"].fillna("Unknown")
test_xgb["Weather"] = test_xgb["Weather"].fillna("Unknown")

# IMPORTANT:
# Use the SAME median used in training
temp_median = data["Temperature"].median()

test_xgb["Temperature"] = (
    test_xgb["Temperature"]
    .fillna(temp_median)
)

# -------------------------
# Latitude Longitude
# -------------------------

def decode_geo(g):
    try:
        lat, lon = pgh.decode(g)
        return pd.Series([lat, lon])
    except:
        return pd.Series([np.nan, np.nan])

test_xgb[["latitude", "longitude"]] = (
    test_xgb["geohash"]
    .apply(decode_geo)
)

# -------------------------
# Cyclic Hour Features
# -------------------------

test_xgb["hour_sin"] = np.sin(
    2 * np.pi * test_xgb["hours"] / 24
)

test_xgb["hour_cos"] = np.cos(
    2 * np.pi * test_xgb["hours"] / 24
)

# -------------------------
# One Hot Encoding
# -------------------------

test_xgb = pd.get_dummies(
    test_xgb,
    columns=["RoadType", "Weather"],
    drop_first=True
)

# -------------------------
# Remove Unused Columns
# -------------------------

test_xgb.drop(
    columns=[
        "Index",
        "timestamp",
        "minute",
        "geohash"
    ],
    inplace=True,
    errors="ignore"
)

# -------------------------
# Match Training Columns
# -------------------------

X_xgb = data.drop("demand", axis=1)

for col in X_xgb.columns:
    if col not in test_xgb.columns:
        test_xgb[col] = 0

test_xgb = test_xgb[X_xgb.columns]

print(test_xgb.shape)
print(test_xgb.head())

(41778, 19)
   day  NumberofLanes  LargeVehicles  Landmarks  Temperature  latitude  \
0   49              1              0          0    16.382587 -5.484924   
1   49              1              0          0     6.476213 -5.484924   
2   49              3              1          1    22.318203 -5.479431   
3   49              2              0          1    16.382587 -5.479431   
4   49              1              0          0    18.266162 -5.479431   

   longitude  hours  time_slot  peak_hour  RoadType_Residential  \
0  90.664673      2          9          0                 False   
1  90.686646      2          9          0                  True   
2  90.653687      2          9          0                  True   
3  90.675659      2          9          0                  True   
4  90.686646      2          9          0                  True   

   RoadType_Street  RoadType_Unknown  Weather_Rainy  Weather_Snowy  \
0            False              True          False          False   


In [169]:
xgb_preds = model2.predict(test_xgb)

In [170]:
import pandas as pd

final_preds = (
    0.8 * test_preds +
    0.2 * xgb_preds
)

submission = pd.DataFrame({
    "Index": test_index,
    "demand": final_preds
})

submission.to_csv(
    "submission3.csv",
    index=False
)

print(submission.head())
print("submission3.csv created successfully")

   Index    demand
0      0  0.052784
1      1  0.032329
2      2  0.022632
3      3  0.025651
4      4  0.068012
submission3.csv created successfully
